# Config 3: frontier model with skills (using Claude)

Config 3 is the live-demo configuration: instead of more parameters or
training, the model gets **tool access** -- it can run PROC CONTENTS /
query `dictionary.columns` through SASPy to get real types, lengths, and
labels instead of guessing. That's a different axis of improvement from
config1 (small local model, zero-shot) and config2 (same base model,
QLoRA-fine-tuned) -- report that asymmetry explicitly rather than present
config3 as "just a smarter model." See the repo root's `PLAN.md` for the
full three-way framing, and `.claude/skills/sas-data-dictionary/SKILL.md`
(repo root) for the complete workflow this notebook runs.

**Important difference from the other two configs' notebooks:** most of
config3's steps here ARE ordinary scripts, but the actual authoring step
(workflow step 4 below) is *you* -- or rather, this Claude Code session --
reading the `.sas` file and writing the dictionary JSON. There is no CLI
for that step; it's not automatable, so that cell is instructions, not
code. Everything else runs the same way config1/config2's notebooks do.

## Setup 1: The model -- this Claude Code session itself

No separate model download or API call to configure for the *writing*
step. If you're following this in Claude Code already (which you are, if
you're reading this notebook that way), skip straight to Setup 2.

**To drive this via the Anthropic API instead** (e.g. to script config3
as a batch job rather than interactively): get a key at
https://console.anthropic.com, `export ANTHROPIC_API_KEY=sk-ant-...` in
your shell (never paste it into a chat session or commit it to this
repo), and write a driver that sends the SAS source + steps 1-3's outputs
to the Messages API with `schema.PROMPT_SCHEMA_BLOCK` as part of the
system prompt (see `../results/llm_judge.py` for a worked example of the
request shape). Record the actual per-call cost from the response's usage
fields for the results table -- don't estimate it. No ready-made script
for this exists in the repo; SETUP.md deliberately leaves it as a stub
for whoever wants config3 to run unattended.

## Setup 2: The skill

The actual workflow lives in `.claude/skills/sas-data-dictionary/SKILL.md`
at the **repo root**, not in this folder -- skills are discovered
relative to the project root. This folder (`config3-frontier-skills/`)
holds the scripts that skill calls (`sas_metadata.py`, `extract.py`,
`header_extract.py`, `validate_dictionary.py`, `write_dictionary.py`,
`push_to_oda.py`, `save_prediction.py`) -- this notebook runs through
those scripts in the same order the skill does.

If you're using Claude Code interactively (not this notebook), the skill
triggers automatically on requests like "document this SAS program."

## Setup 3: Python dependencies

`saspy` + `pandas`, needed only for the ground-truth harvest (workflow
step 1) and the ODA push (workflow step 6) -- the authoring step itself
needs nothing beyond Claude reading the file.

In [ ]:
!pip install -r requirements.txt

### Which interpreter runs the SASPy steps

Steps 1 and 6 below are the only ones that talk to SAS, and they need
`saspy` + `pandas`. On this box those live in a separate venv
(`/internal/venvs/main`) rather than in the kernel running this notebook,
which is why those two cells used to hardcode that path. That path does
not exist anywhere else, so the cell below picks whichever interpreter
actually has `saspy` importable -- the venv when it's there, otherwise
this notebook's own kernel (after the `pip install` above).

In [ ]:
import os
import subprocess
import sys

def _has_saspy(python):
    try:
        return subprocess.run([python, "-c", "import saspy"],
                              capture_output=True).returncode == 0
    except OSError:
        return False

VENV_PY = "/internal/venvs/main/bin/python3"
PY = VENV_PY if _has_saspy(VENV_PY) else sys.executable
os.environ["PY"] = PY
print("SASPy steps will run under:", PY)
if not _has_saspy(PY):
    print("  ...which cannot import saspy yet. Run the pip cell above, or use the "
          "venv that has it. Steps 2-5 and 7 do not need saspy and work regardless.")

## Setup 4: SAS OnDemand for Academics (ODA) credentials

This is the step config3 benefits from MOST -- it supplies the ground
truth (`dictionary.columns`/`dictionary.tables`) that gives config3 its
accuracy advantage over configs 1/2. Config3 still runs without it
(falls back to the static source scan only, same as configs 1/2 have),
but then the "tool access" story the plan wants to tell isn't actually
being exercised -- do this if you want a real config3 result, not just a
config1-equivalent with extra steps.

**Get a free ODA account** first if you don't have one (SAS OnDemand for
Academics signup -- no cost, academic/non-commercial use).

**Create `~/.authinfo` yourself, in your own terminal** -- not in a
notebook cell that gets saved:
```bash
echo "oda user YOUR_ODA_EMAIL password YOUR_ODA_PASSWORD" >> ~/.authinfo
chmod 600 ~/.authinfo
```
Then check your ODA region matches `config/sascfg_personal.py`'s
`iomhost` list (already filled in for US-region/usw2 -- see config1's
`SETUP.md` for the other two regions' host names if your account
differs). No Java install needed -- the portable JRE at the repo root
(`../jre/`) is picked up automatically.

In [ ]:
!test -f ~/.authinfo && echo "~/.authinfo found" || echo "missing -- create it in a terminal first, see the cell above"

---
## The workflow (SKILL.md's 7 steps)

Demonstrated below on one eval program, `prog900_estab.sas`, the same one
config1's notebook uses -- so all three configs' notebooks produce
directly comparable output for the same program. For a whole directory,
loop steps 1-5 per file (there's no single CLI that does steps 1/4/5
together, since step 4 is you reading and writing, not a subprocess) and
run step 6 once at the end against the accumulated catalog -- see
`SKILL.md`'s "Batch use" section.

In [ ]:
# Both a Python name and an env var: the `!` cells below expand $PROGRAM /
# $SAS_FILE / $PY, and IPython only substitutes names it can find in this
# namespace -- if any one of them is missing it leaves the whole command
# for the shell instead, which then needs them in the environment. Setting
# both keeps either path working.
import os

PROGRAM = "prog900_estab"
SAS_FILE = "../eval-programs/programs/prog900_estab.sas"
os.environ["PROGRAM"] = PROGRAM
os.environ["SAS_FILE"] = SAS_FILE
print(PROGRAM, "->", SAS_FILE)

### 1. Gather ground truth from SAS itself

The anti-hallucination backbone -- real column names, types, lengths,
formats, informats, and labels, straight from SAS's own
`dictionary.columns`/`dictionary.tables`. Needs a working SASPy
connection (Setup 4 above). If genuinely no SAS session is reachable,
you can still proceed to step 4 on the static scan alone, but say so
explicitly in the output -- the dictionary is meaningfully less
trustworthy without this step, not equally good.

In [ ]:
!$PY sas_metadata.py $SAS_FILE --out /tmp/column_metadata.json

### 2. Gather human-authored context (often nothing -- that's expected)

Pulls a leading header-comment block's fields plus any inline
`/* comment */` glosses sitting on variable-defining lines.
`header_found: false` is a normal, common result -- this project's own
eval set has zero header comments on purpose, matching what config1/
config2 see too.

In [ ]:
!python3 header_extract.py $SAS_FILE

### 3. Gather the static identifier allow-list

Regex scan of the source for every dataset/macro-param/variable-like
name that demonstrably appears in the text -- the fallback allow-list
when ground truth is incomplete, and the only source for macro
parameter names (SAS metadata tables have no concept of those).

In [ ]:
!python3 extract.py $SAS_FILE

### 4. Read the source, then author the dictionary yourself -- NOT SCRIPTABLE

**There is nothing to run in this cell.** This is the step that makes
config3 config3: ask this Claude Code session (or whichever
frontier-model session you're running the skill in) to read
`$SAS_FILE` and combine it with steps 1-3's output above into a
dictionary JSON matching `schema.py`'s shape -- e.g. prompt it with
"document `eval-programs/programs/prog900_estab.sas` using the
sas-data-dictionary skill."

Hard rules it follows while writing (full detail in `SKILL.md`):
- never name a dataset/variable/macro-param that isn't in the ground
  truth or the static scan;
- prefer a real `label` from ground truth over a guessed one;
- `type`/`length`/`label` come from ground truth when available;
- one `variable_dictionary` row per variable per *output* dataset, not
  every intermediate `_tmp` step;
- `macro_reference[*].positional_params` vs `keyword_params` split on
  whether the `%macro` signature gives that parameter a default.

Save the result as `/tmp/dictionary.json` (the path step 5 below
expects) before continuing.

### 5. Validate, then persist to the local catalog

Cross-checks every identifier in `/tmp/dictionary.json` against ground
truth + the static scan, stamps any misses as `guardrail_flagged`, and
upserts into the local JSONL catalog. If it reports flags, don't just
push anyway -- go back and check whether you mis-typed a real
identifier or actually invented one, and fix the JSON before re-running
this cell.

In [ ]:
!python3 write_dictionary.py \
    --program-name $PROGRAM --source $SAS_FILE \
    --dictionary /tmp/dictionary.json \
    --column-metadata /tmp/column_metadata.json \
    --catalog catalog/

### 6. Push to SAS as real datasets

Writes/replaces `PROGRAM_SUMMARY`, `MACRO_PARAMS`, `DATA_DICTIONARY`,
and `COLUMN_METADATA` in the target SAS library (default `SASUSER` on
ODA -- pass `--libname`/`--libpath` for a different, permanent,
custom-path library instead). Each run mirrors the CURRENT full
contents of the local catalog, so it's safe to re-run after documenting
more programs -- it doesn't append duplicates. Optional: skip if you
only want the local JSON/catalog output.

In [ ]:
!$PY push_to_oda.py --catalog catalog/

### 7. If scoring this run against the eval set

Drops your authored JSON into the shape `results/score.py` expects
(same `.pred.json`/`.meta.json` layout config1 and config2 write, at
`../results/preds/config3-frontier-skills` -- matching the convention
both of their notebooks also use) so `results/run_eval.py` scores all
three configs identically. Record `--elapsed-sec` honestly -- fill in
how long steps 1-5 actually took for this program.

In [ ]:
# --elapsed-sec is NOT optional bookkeeping: replace 0 with how long steps
# 1-5 actually took for this program (wall clock, honestly measured). The
# 2026-09-17 run filled in a flat 90.0 for all 20 programs; the results
# table now detects an identical-for-every-program time and prints it as
# "(placeholder*)" rather than as a measurement.
#
# Add --cost-usd only if you drove config3 through the Anthropic API and
# have the real number from the response's usage fields. Left off, the
# table prints "not recorded*" -- which is the honest cell for an
# interactive session, and is NOT the same as $0.
!python3 save_prediction.py --program-name $PROGRAM \
    --dictionary /tmp/dictionary.json --elapsed-sec 0 \
    --used-proc-contents \
    --out ../results/preds/config3-frontier-skills

Once all 20 programs have a `.pred.json`, score the run. `run_eval.py`
stamps a `.provenance.json` sidecar recording exactly which eval corpus
was scored, and `--table` marks a row **STALE** rather than printing
numbers that no longer refer to the corpus on disk. That check exists
because config3's own 2026-09-17 run was scored against a corpus that was
then rewritten, and read `1.00` on every metric while the same
predictions against the current gold read `0.79` -- see
`../results/outputs/stale-2026-09-17/README.md`.

Config3 is also the config the plan says to score twice: once as run
(with `sas_metadata.py` ground truth available), and once *without* it, as
a fairer isolate of model quality. Pass the harvested metadata directory
as `--extra-source-dir` for the first, per `PLAN.md` section 3's "For
config 3, PROC CONTENTS output also counts as a valid source".

In [ ]:
!python3 ../results/run_eval.py --config config3-frontier-skills \
    --pred-dir ../results/preds/config3-frontier-skills \
    --out-prefix ../results/outputs/config3-frontier-skills

In [ ]:
!python3 ../results/run_eval.py --table --rows ../results/rows.example.json

## The asymmetry to report honestly

Config 3's PROC CONTENTS / `dictionary.columns` access is real ground
truth the other two configs structurally cannot get (config1 has no SAS
connection at all by design; config2's fine-tuning happens offline, with
no live SAS session at inference time either). If config3 scores much
higher on `type_length_accuracy` or `hallucination_rate`, that may be
measuring "has ground truth" more than "is a better model" -- say so in
the writeup, and consider also reporting a config3-without-ground-truth
run (skip step 1, `write_dictionary.py` with no `--column-metadata`) as
a fairer isolate of model quality alone.

## Don't confuse this with config1

`config1-gemma-cpu/document_sas.py` is a separate, independent pipeline
(local Ollama model, JSON-mode prompting, its own regex guardrail, no
ground-truth SAS metadata at all) used to establish the CPU/air-gapped
floor for comparison. Don't mix the two catalogs by hand-editing; they
write to different `catalog/` directories under their own config
folders by design, specifically so config1 and config3 runs never
clobber each other.